# Q35: What distinguishes Linear Filters from Non-Linear Filters?
**Topic:** Computer-vision | **Difficulty:** Medium | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
What distinguishes Linear Filters from Non-Linear Filters?

## Detailed Answer

### Linear Filters
A **linear filter** applies a weighted sum (convolution) of pixel values:

$$g(x,y) = \sum_{i}\sum_{j} K(i,j) \cdot f(x+i, y+j)$$

**Properties:**
- Satisfy superposition: $F(a \cdot I_1 + b \cdot I_2) = a \cdot F(I_1) + b \cdot F(I_2)$
- Can be expressed as matrix multiplication
- Implemented efficiently via FFT (convolution theorem)

**Examples:**
| Filter | Purpose | Kernel |
|--------|---------|--------|
| Box/Mean | Smoothing | All 1s, normalized |
| Gaussian | Smoothing (weighted) | Gaussian distribution |
| Sobel | Edge detection | Gradient approximation |
| Laplacian | Edge detection | 2nd derivative |
| Sharpening | Enhance edges | Center-weighted |

**Limitation:** Cannot effectively remove salt-and-pepper noise (impulse noise) — they blur edges.

### Non-Linear Filters
A **non-linear filter** applies a non-linear function to pixel neighborhoods:

**Examples:**
| Filter | Operation | Strength |
|--------|-----------|----------|
| Median | Median of neighborhood | Removes impulse noise, preserves edges |
| Bilateral | Weighted by spatial + intensity distance | Smooths while preserving edges |
| Min/Max (Morphological) | Min or Max in neighborhood | Erosion/Dilation |
| Non-Local Means | Weighted average of similar patches | Best denoising quality |

**Key Difference:**
| Property | Linear | Non-Linear |
|----------|--------|------------|
| Superposition | Yes | No |
| Edge preservation | Poor | Good |
| Impulse noise removal | Poor | Excellent (median) |
| Computational cost | Lower (FFT) | Higher |
| Mathematical analysis | Easier | Harder |
| Frequency domain | Applicable | Not applicable |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def apply_mean_filter(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """Linear mean (box) filter."""
    pad = kernel_size // 2
    padded = np.pad(image, pad, mode='edge')
    result = np.zeros_like(image, dtype=float)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]
            result[i, j] = np.mean(region)
    return result


def apply_median_filter(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """Non-linear median filter."""
    pad = kernel_size // 2
    padded = np.pad(image, pad, mode='edge')
    result = np.zeros_like(image, dtype=float)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+kernel_size, j:j+kernel_size]
            result[i, j] = np.median(region)
    return result


# Create test image with salt-and-pepper noise
np.random.seed(42)
image = np.zeros((50, 50))
image[10:40, 10:40] = 200  # white square on black background

# Add salt-and-pepper noise
noisy = image.copy()
noise_mask = np.random.random(image.shape)
noisy[noise_mask < 0.05] = 0    # pepper
noisy[noise_mask > 0.95] = 255  # salt

# Apply filters
mean_filtered = apply_mean_filter(noisy, 3)
median_filtered = apply_median_filter(noisy, 3)

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
    [image, noisy, mean_filtered, median_filtered],
    ['Original', 'Salt & Pepper Noise', 'Mean Filter (Linear)', 'Median Filter (Non-Linear)']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("Notice: Median filter removes noise while preserving sharp edges!")
print("Mean filter blurs the edges while only partially reducing noise.")

## Key Takeaways
- **Linear filters** (mean, Gaussian) are fast but blur edges — good for Gaussian noise
- **Median filter** is the go-to for salt-and-pepper / impulse noise — preserves edges
- **Bilateral filter** is the best of both worlds — smooths noise while keeping edges
- In deep learning, conv layers are linear filters — non-linearity comes from activation functions